In [494]:
import pandas as pd
import boto3
from io import BytesIO
import ast
import numpy as np

In [495]:
bucket_name = "vasantha-leela-2026-12345"
s3 = boto3.client("s3")

In [496]:
response = s3.list_objects_v2(Bucket=bucket_name, Prefix="Dataset")

for obj in response.get("Contents"):
    print(obj["Key"])

Dataset/bangalore_cars.xlsx
Dataset/chennai_cars.xlsx
Dataset/delhi_cars.xlsx
Dataset/hyderabad_cars.xlsx
Dataset/jaipur_cars.xlsx
Dataset/kolkata_cars.xlsx


In [497]:
files = [
    "Dataset/bangalore_cars.xlsx",
    "Dataset/chennai_cars.xlsx",
    "Dataset/delhi_cars.xlsx",
    "Dataset/hyderabad_cars.xlsx",
    "Dataset/jaipur_cars.xlsx",
    "Dataset/kolkata_cars.xlsx",
]

In [498]:
dataframes = []

for file in files:
    response = s3.get_object(Bucket=bucket_name, Key=file)

    data = response["Body"].read()

    df = pd.read_excel(file)

    city = file.split("/")[-1].replace("_cars.xlsx", "").strip()
    df["city"] = city

    dataframes.append(df)

    print(city, df.shape)

bangalore (1481, 6)
chennai (1419, 6)
delhi (1485, 6)
hyderabad (1483, 6)
jaipur (1120, 6)
kolkata (1381, 6)


In [499]:
raw = pd.concat(dataframes, ignore_index=True)

In [500]:
raw.shape 

(8369, 6)

In [501]:
raw.head()

,new_car_detail,new_car_overview,new_car_feature,new_car_specs,car_links,city
0,"{'it': 0, 'ft': 'Petrol', 'bt': 'Hatchback', '...","{'heading': 'Car overview', 'top': [{'key': 'R...","{'heading': 'Features', 'top': [{'value': 'Pow...","{'heading': 'Specifications', 'top': [{'key': ...",https://www.cardekho.com/used-car-details/used...,bangalore
1,"{'it': 0, 'ft': 'Petrol', 'bt': 'SUV', 'km': '...","{'heading': 'Car overview', 'top': [{'key': 'R...","{'heading': 'Features', 'top': [{'value': 'Pow...","{'heading': 'Specifications', 'top': [{'key': ...",https://www.cardekho.com/buy-used-car-details/...,bangalore
2,"{'it': 0, 'ft': 'Petrol', 'bt': 'Hatchback', '...","{'heading': 'Car overview', 'top': [{'key': 'R...","{'heading': 'Features', 'top': [{'value': 'Pow...","{'heading': 'Specifications', 'top': [{'key': ...",https://www.cardekho.com/used-car-details/used...,bangalore
3,"{'it': 0, 'ft': 'Petrol', 'bt': 'Sedan', 'km':...","{'heading': 'Car overview', 'top': [{'key': 'R...","{'heading': 'Features', 'top': [{'value': 'Pow...","{'heading': 'Specifications', 'top': [{'key': ...",https://www.cardekho.com/buy-used-car-details/...,bangalore
4,"{'it': 0, 'ft': 'Diesel', 'bt': 'SUV', 'km': '...","{'heading': 'Car overview', 'top': [{'key': 'R...","{'heading': 'Features', 'top': [{'value': 'Pow...","{'heading': 'Specifications', 'top': [{'key': ...",https://www.cardekho.com/used-car-details/used...,bangalore


In [502]:
raw["city"].value_counts()

city
delhi        1485
hyderabad    1483
bangalore    1481
chennai      1419
kolkata      1381
jaipur       1120
Name: count, dtype: int64

In [503]:
def extact(value):
    data = ast.literal_eval(value)

    return pd.Series(
        {
            "fuel_type": data.get("ft"),
            "body_type": data.get("bt"),
            "km_driven": data.get("km"),
            "transmission": data.get("transmission"),
            "owner_number": data.get("ownerNo"),
            "owner": data.get("owner"),
            "brand": data.get("oem"),
            "model": data.get("model"),
            "model_year": data.get("modelYear"),
            "variant": data.get("variantName"),
            "price_raw": data.get("price"),
        }
    )

In [504]:
detaildf = raw["new_car_detail"].apply(extact)

In [505]:
detaildf.head()

,fuel_type,body_type,km_driven,transmission,owner_number,owner,brand,model,model_year,variant,price_raw
0,Petrol,Hatchback,"1,20,000",Manual,3,3rd Owner,Maruti,Maruti Celerio,2015,VXI,₹ 4 Lakh
1,Petrol,SUV,"32,706",Manual,2,2nd Owner,Ford,Ford Ecosport,2018,1.5 Petrol Titanium BSIV,₹ 8.11 Lakh
2,Petrol,Hatchback,"11,949",Manual,1,1st Owner,Tata,Tata Tiago,2018,1.2 Revotron XZ,₹ 5.85 Lakh
3,Petrol,Sedan,"17,794",Manual,1,1st Owner,Hyundai,Hyundai Xcent,2014,1.2 Kappa S Option,₹ 4.62 Lakh
4,Diesel,SUV,"60,000",Manual,1,1st Owner,Maruti,Maruti SX4 S Cross,2015,DDiS 200 Zeta,₹ 7.90 Lakh


In [506]:
clean = pd.concat([raw[["city", "car_links"]], detaildf], axis=1)

In [507]:
def extract(value):
    data = ast.literal_eval(value)

    result = {}

    for i in data["top"]:
        key = i["key"]
        val = i["value"]

        result[key] = val

    return pd.Series(result)

In [508]:
overviewdf = raw["new_car_overview"].apply(extract)

overviewdf.head()

,Registration Year,Insurance Validity,Fuel Type,Seats,Kms Driven,RTO,Ownership,Engine Displacement,Transmission,Year of Manufacture
0,2015,Third Party insurance,Petrol,5 Seats,"1,20,000 Kms",KA51,Third Owner,998 cc,Manual,2015.0
1,Feb 2018,Comprehensive,Petrol,5 Seats,"32,706 Kms",KA05,Second Owner,1497 cc,Manual,2018.0
2,Sept 2018,Comprehensive,Petrol,5 Seats,"11,949 Kms",KA03,First Owner,1199 cc,Manual,2018.0
3,Dec 2014,Comprehensive,Petrol,5 Seats,"17,794 Kms",KA53,First Owner,1197 cc,Manual,2014.0
4,2015,Third Party insurance,Diesel,5 Seats,"60,000 Kms",KA04,First Owner,1248 cc,Manual,2015.0


In [509]:
overview_df = overviewdf.rename(
    columns={
        "Registration Year": "registration_year",
        "Insurance Validity": "insurance_validity",
        "Fuel Type": "fuel_type_overview",
        "Seats": "seats",
        "Kms Driven": "kms_driven_overview",
        "RTO": "rto",
        "Ownership": "ownership",
        "Engine Displacement": "engine_displacement",
        "Transmission": "transmission_overview",
        "Year of Manufacture": "year_of_manufacture",
    }
)

In [510]:
overview_df.shape

(8369, 10)

In [511]:
overview_df.head()

,registration_year,insurance_validity,fuel_type_overview,seats,kms_driven_overview,rto,ownership,engine_displacement,transmission_overview,year_of_manufacture
0,2015,Third Party insurance,Petrol,5 Seats,"1,20,000 Kms",KA51,Third Owner,998 cc,Manual,2015.0
1,Feb 2018,Comprehensive,Petrol,5 Seats,"32,706 Kms",KA05,Second Owner,1497 cc,Manual,2018.0
2,Sept 2018,Comprehensive,Petrol,5 Seats,"11,949 Kms",KA03,First Owner,1199 cc,Manual,2018.0
3,Dec 2014,Comprehensive,Petrol,5 Seats,"17,794 Kms",KA53,First Owner,1197 cc,Manual,2014.0
4,2015,Third Party insurance,Diesel,5 Seats,"60,000 Kms",KA04,First Owner,1248 cc,Manual,2015.0


In [512]:
def extract_specs(value):
    data = ast.literal_eval(value)

    result = {}

    for i in data["top"]:
        key = i["key"]
        val = i["value"]

        result[key] = val

    return pd.Series(result)

In [513]:
specsdf = raw["new_car_specs"].apply(extract_specs)

specsdf.head()

,Mileage,Engine,Max Power,Torque,Seats,Wheel Size
0,23.1 kmpl,998 CC,67.04bhp,90Nm,5,NaN
1,17 kmpl,1497 CC,121.31bhp,150Nm,5,16
2,23.84 kmpl,1199 CC,84bhp,114Nm,5,14
3,19.1 kmpl,1197 CC,81.86bhp,113.75Nm,5,14
4,23.65 kmpl,1248 CC,88.5bhp,200Nm,5,16


In [514]:
specs_df = specsdf.rename(
    columns={
        "Mileage": "mileage",
        "Engine": "engine",
        "Max Power": "max_power",
        "Torque": "torque",
        "Seats": "seats_specs",
    }
)

In [515]:
specs_df.head()

,mileage,engine,max_power,torque,seats_specs,Wheel Size
0,23.1 kmpl,998 CC,67.04bhp,90Nm,5,NaN
1,17 kmpl,1497 CC,121.31bhp,150Nm,5,16
2,23.84 kmpl,1199 CC,84bhp,114Nm,5,14
3,19.1 kmpl,1197 CC,81.86bhp,113.75Nm,5,14
4,23.65 kmpl,1248 CC,88.5bhp,200Nm,5,16


In [516]:
def extract_feature(value):
    data = ast.literal_eval(value)

    features = []

    for i in data["data"]:
        for j in i["list"]:
            features.append(j.get("value"))

    return ", ".join(features)

In [517]:
featuresdf = raw["new_car_feature"].apply(extract_feature)

In [518]:
# featuresdf=featuresdf.to_frame(name="features")

featuresdf.shape

(8369,)

In [519]:
cleanedf = pd.concat(
    [
        detaildf,
        overview_df,
        featuresdf,
        specs_df,
        raw[["city", "car_links"]]
    ],
    axis=1
)

In [520]:
overview_seats = pd.to_numeric(
    cleanedf["seats"].astype("string").str.extract(r"(\d+)")[0],
    errors="coerce"
)

spec_seats = pd.to_numeric(
    cleanedf["seats_specs"].astype("string").str.extract(r"(\d+)")[0],
    errors="coerce"
)

cleanedf["seats"] = overview_seats.fillna(spec_seats)

cleanedf = cleanedf.drop(columns=["seats_specs"])

In [521]:
cleanedf.shape

(8369, 29)

In [522]:
cleanedf.columns.tolist()

['fuel_type',
 'body_type',
 'km_driven',
 'transmission',
 'owner_number',
 'owner',
 'brand',
 'model',
 'model_year',
 'variant',
 'price_raw',
 'registration_year',
 'insurance_validity',
 'fuel_type_overview',
 'seats',
 'kms_driven_overview',
 'rto',
 'ownership',
 'engine_displacement',
 'transmission_overview',
 'year_of_manufacture',
 'new_car_feature',
 'mileage',
 'engine',
 'max_power',
 'torque',
 'Wheel Size',
 'city',
 'car_links']

In [523]:
print("Seats missing:", cleanedf["seats"].isnull().sum())

print("Seats data type:", cleanedf["seats"].dtype)

display(cleanedf[["seats"]].head())

Seats missing: 6
Seats data type: Int64


,seats
0,5
1,5
2,5
3,5
4,5


In [524]:
cleanedf["seats"] = pd.to_numeric(
    cleanedf["seats"],
    errors="coerce"
)

In [525]:
print("Data type:", cleanedf["seats"].dtype)

print("Missing values:", cleanedf["seats"].isnull().sum())

display(cleanedf[["seats"]].head(10))

Data type: Int64
Missing values: 6


,seats
0,5
1,5
2,5
3,5
4,5
5,5
6,5
7,5
8,5
9,5


In [526]:
(cleanedf["fuel_type"] == cleanedf["fuel_type_overview"]).value_counts() 

True     8255
False     114
Name: count, dtype: int64

In [527]:
cleanedf[["km_driven", "kms_driven_overview"]].isnull().sum()

km_driven              0
kms_driven_overview    2
dtype: int64

In [528]:
cleanedf[["transmission", "transmission_overview"]].isnull().sum()

transmission             0
transmission_overview    0
dtype: int64

In [529]:
cleanedf[["owner", "ownership"]].isnull().sum()

owner         0
ownership    32
dtype: int64

In [530]:
cleanedf["registration_year"] = cleanedf["registration_year"].str.extract(r"(\d{4})")[0]

In [531]:
cleanedf["registration_year"] = pd.to_numeric(cleanedf["registration_year"], errors="coerce")

In [532]:
cleanedf["engine"] = cleanedf["engine"].str.extract(r"([\d.]+)")[0]

cleanedf["engine"] = pd.to_numeric(cleanedf["engine"], errors="coerce")

In [533]:
cleanedf["mileage"] = cleanedf["mileage"].str.extract(r"([\d.]+)")[0]

cleanedf["mileage"] = pd.to_numeric(cleanedf["mileage"], errors="coerce")

In [534]:
cleanedf["max_power"] = cleanedf["max_power"].str.extract(r"([\d.]+)")[0]

cleanedf["max_power"] = pd.to_numeric(cleanedf["max_power"], errors="coerce")

In [535]:
cleanedf["torque"] = cleanedf["torque"].str.extract(r"([\d.]+)")[0]

cleanedf["torque"] = pd.to_numeric(cleanedf["torque"], errors="coerce")

In [536]:
cleanedf["price_lakh"] = (
    cleanedf["price_raw"]
    .str.replace("₹", "", regex=False)
    .str.replace("Lakh", "", regex=False)
    .str.strip()
)

cleanedf["price_lakh"] = pd.to_numeric(cleanedf["price_lakh"], errors="coerce")

In [537]:
cleanedf = cleanedf.drop(columns=["year_of_manufacture", "engine_displacement"])

In [538]:
cleanedf = cleanedf.drop(columns=["owner"])

In [539]:
cleanedf.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
fuel_type,8369,5,Petrol,5554,NaN,NaN,NaN,NaN,NaN,NaN,NaN
body_type,8369,11,Hatchback,3567,NaN,NaN,NaN,NaN,NaN,NaN,NaN
km_driven,8369,4090,"1,20,000",315,NaN,NaN,NaN,NaN,NaN,NaN,NaN
transmission,8369,2,Manual,6052,NaN,NaN,NaN,NaN,NaN,NaN,NaN
owner_number,8369.0,NaN,NaN,NaN,1.360139,0.641958,0.0,1.0,1.0,2.0,5.0
brand,8369,33,Maruti,2252,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model,8369,315,Honda City,370,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model_year,8369.0,NaN,NaN,NaN,2016.503286,3.921465,1985.0,2014.0,2017.0,2019.0,2023.0
variant,8369,2157,VXI,337,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price_raw,8369,1400,₹ 6.50 Lakh,106,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [540]:
cleanedf[cleanedf["rto"].isna()][
    ["city", "brand", "model", "rto"]
].head(20)

,city,brand,model,rto
10,bangalore,Maruti,Maruti Baleno,NaN
20,bangalore,BMW,BMW 5 Series,NaN
24,bangalore,Mercedes-Benz,Mercedes-Benz S-Class,NaN
27,bangalore,Hyundai,Hyundai Santro Xing,NaN
33,bangalore,Hyundai,Hyundai Xcent,NaN
53,bangalore,Maruti,Maruti Alto,NaN
73,bangalore,Maruti,Maruti Ciaz,NaN
75,bangalore,Renault,Renault Lodgy,NaN
121,bangalore,Hyundai,Hyundai i10,NaN
124,bangalore,Hyundai,Hyundai i10,NaN


In [541]:
cleanedf["rto"].value_counts(dropna=False).head(20)

rto
NaN     888
WB02    452
RJ14    367
TS07    282
TS08    244
KA03    244
RJ45    217
HR26    195
KA05    183
TS09    167
WB06    164
KA01    155
KA51    155
KA04    135
TN07    132
TN02    118
UP16    115
TN09    112
KA53    106
WB08    102
Name: count, dtype: int64

In [542]:
display(
    cleanedf[cleanedf["torque"] > 500]
    [["brand", "model", "engine", "max_power", "torque", "price_lakh"]]
    .sort_values("torque", ascending=False)
)

,brand,model,engine,max_power,torque,price_lakh
3383,Mercedes-Benz,Mercedes-Benz AMG G 63,3982.0,576.63,850.0,NaN
3199,Mercedes-Benz,Mercedes-Benz AMG G 63,3982.0,576.63,850.0,NaN
2134,Audi,Audi Q7,4134.0,335.20,800.0,15.00
5508,Mercedes-Benz,Mercedes-Benz EQC,0.0,402.30,760.0,75.00
1612,Mercedes-Benz,Mercedes-Benz S-Class,4663.0,459.00,700.0,NaN
...,...,...,...,...,...,...
1049,Jaguar,Jaguar XJ,5000.0,379.70,515.0,17.00
1050,Jaguar,Jaguar XJ,5000.0,379.70,515.0,17.00
5373,Mercedes-Benz,Mercedes-Benz M-Class,2987.0,165.00,510.0,17.25
4722,Mercedes-Benz,Mercedes-Benz M-Class,2987.0,165.00,510.0,12.00


In [543]:
cleanedf["km_driven"] = pd.to_numeric(
    cleanedf["km_driven"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)

In [544]:
print("KM > 300000:")
display(
    cleanedf[cleanedf["km_driven"] > 300000]
    [["brand", "model", "model_year", "km_driven", "price_lakh"]]
    .sort_values("km_driven", ascending=False)
)

KM > 300000:


,brand,model,model_year,km_driven,price_lakh
1539,Mercedes-Benz,Mercedes-Benz C-Class,2018,5500000,36.00
2679,Toyota,Toyota Innova Crysta,2017,2000022,17.50
1996,Mahindra,Mahindra Scorpio,2019,1200000,11.50
7610,Kia,Kia Sonet,2021,975000,9.75
1538,Toyota,Toyota Innova Crysta,2018,800000,21.50


In [545]:
print("Mileage > 40:")
display(
    cleanedf[cleanedf["mileage"] > 40]
    [["brand", "model", "mileage", "engine", "price_lakh"]]
    .sort_values("mileage", ascending=False)
)

Mileage > 40:


,brand,model,mileage,engine,price_lakh
839,Mahindra,Mahindra e2o Plus,140.0,72.0,4.6
574,Mahindra,Mahindra e2o Plus,110.0,NaN,5.5


In [546]:
print("KM > 300000:")
display(
    cleanedf[cleanedf["km_driven"] > 300000]
    [["brand", "model", "model_year", "km_driven", "price_lakh"]]
    .sort_values("km_driven", ascending=False)
)

KM > 300000:


,brand,model,model_year,km_driven,price_lakh
1539,Mercedes-Benz,Mercedes-Benz C-Class,2018,5500000,36.00
2679,Toyota,Toyota Innova Crysta,2017,2000022,17.50
1996,Mahindra,Mahindra Scorpio,2019,1200000,11.50
7610,Kia,Kia Sonet,2021,975000,9.75
1538,Toyota,Toyota Innova Crysta,2018,800000,21.50


In [547]:
print("Mileage > 40:")
display(
    cleanedf[cleanedf["mileage"] > 40]
    [["brand", "model", "mileage", "engine", "price_lakh"]]
    .sort_values("mileage", ascending=False)
)

Mileage > 40:


,brand,model,mileage,engine,price_lakh
839,Mahindra,Mahindra e2o Plus,140.0,72.0,4.6
574,Mahindra,Mahindra e2o Plus,110.0,NaN,5.5


In [548]:
numeric_cols = [
    "km_driven",
    "owner_number",
    "model_year",
    "registration_year",
    "mileage",
    "engine",
    "max_power",
    "torque",
    "seats",
    "price_lakh"
]

for col in numeric_cols:

    values = (
        cleanedf[col]
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.strip()
        .str.lower()
    )

    values = values.replace({
        "null": pd.NA,
        "none": pd.NA,
        "nan": pd.NA,
        "": pd.NA
    })

    cleanedf[col] = pd.to_numeric(
        values,
        errors="coerce"
    )

In [549]:
print("Shape:", cleanedf.shape)

print("\nMissing values:")
print(cleanedf.isnull().sum().sum())

print("\nDuplicate rows:")
print(cleanedf.duplicated().sum())

Shape: (8369, 27)

Missing values:
4508

Duplicate rows:
0


In [550]:
cleanedf.to_csv("Cleaned_cars.csv",index=False)